# ODI to Databricks Migration: TRG_DEP_INSERT

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook migrates an ODI INSERT statement for the `TRG_DEP` table from Oracle to Databricks Spark SQL. It includes standard ETL parameter widgets, DDL for the target table, and the converted INSERT logic.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Target Table DDL

In [ ]:
%sql
-- Drop target table if it exists to ensure a clean create or full load scenario.
-- This step typically precedes a full reload or schema recreation.
DROP TABLE IF EXISTS workspace.hr.trg_dep;

In [ ]:
%sql
-- Create the target table 'trg_dep' in the 'hr' schema.
-- Data types are mapped from Oracle (e.g., NUMBER(p,0) to BIGINT, VARCHAR2 to STRING).
CREATE TABLE workspace.hr.trg_dep (
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT
) USING DELTA;

## Insert into Target

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
-- Insert data from the source 'departments' table into the target 'trg_dep' table.
-- Oracle's /*+ APPEND PARALLEL */ hint is removed as it's not applicable in Spark SQL.
INSERT INTO workspace.hr.trg_dep
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID
)
SELECT
    DEPARTMENTS.DEPARTMENT_ID,
    DEPARTMENTS.DEPARTMENT_NAME,
    DEPARTMENTS.MANAGER_ID,
    DEPARTMENTS.LOCATION_ID
FROM
    workspace.hr.departments AS DEPARTMENTS;

## Validation

In [ ]:
%sql
-- Verify the total number of records inserted into the target table.
SELECT COUNT(*) AS total_records_trg_dep
FROM workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Source Table `workspace.hr.departments`:** This notebook assumes that `workspace.hr.departments` already exists and contains data. If not, its DDL and data loading would be required before running this notebook.
2.  **Data Type Assumptions:** The DDL for `workspace.hr.trg_dep` infers data types based on common Oracle `DEPARTMENTS` table structures. Review `BIGINT` and `STRING` mappings to ensure they align with the exact precision and scale of the original Oracle source columns.
3.  **Error Handling/Logging:** The original ODI process likely had extensive error handling and logging (e.g., E$ tables, `SNP_CHECK_TAB`). This conversion focuses solely on the SQL logic. Implement Databricks-native error handling and logging mechanisms if required for production.
4.  **Performance:** The `/*+ APPEND PARALLEL */` hint from Oracle was removed. Databricks Delta Lake automatically optimizes writes and parallel execution. For larger datasets, consider `OPTIMIZE` and `ZORDER BY` on relevant columns for read performance, if not already specified in the source.
5.  **Incremental Logic:** This ODI snippet represents a direct insert, likely part of a full load or a specific sub-process. If the original ODI scenario involved incremental loading, further logic (e.g., date-based filtering, change data capture) would need to be added. No incremental logic was present in the provided source SQL.